In [79]:
import requests

URL = "https://mosmap.ru/api/api_analitic.php"
API_KEY = "b089de13-5470-4717-9d5f-425f2b4b41a8"

In [80]:
def api_call(lat, lon, radius):
    params = {
        'apikey': API_KEY,
        'longitude': lon,
        'latitude': lat,
        'radius': radius
    }
    
    response = requests.get(URL, params=params)
    data = response.json()
    return data

In [64]:
%%time
data = api_call(55.741036, 37.654256, 300)

CPU times: total: 2.61 s
Wall time: 14.2 s


In [85]:
import re
import numpy as np
from functools import lru_cache


needed_keys = [
    'latitude', 'longitude',
    'district_name', # название района
    'price', #
    'orgs', # количество организаций
    'zone', # Информация по жилой зоне вокруг точки
    'bcenters', # Расстояние до ближайших бизнес-центров (в метрах) (нужно агрегировать)
    'metro_exits', # Выходы метро и координаты до него
    'traffic1', 'traffic2', 'traffic3', 'traffic4' # 4 типа трафика
]

@lru_cache(maxsize=2048)
def get_data(lat, lon, radius):
    data = api_call(lat, lon, radius)

    # Организации: количество вокруг
    orgs = [d for d in data["orgs"].values()]
    orgs_dict = {f"{d['group_name']}_count_{radius}m": d['count'] for d in orgs}

    # Зона: информация о недвижимости вокруг
    zone_info = {d['name']: d['value'] for d in data["zone"]}
    n_buildings = zone_info['Строений']
    n_living_buildings = zone_info['Жилых домов']
    n_flats = zone_info['Квартир']
    avg_building_age = int(zone_info['Средний возраст домов'].replace(" лет", ""))
    secondary_flat_price = float(zone_info['Стоимость метра жилья вторичка'].replace(" руб.", ""))

    # Бизнес-центры -- расстояния
    bc_distances = [d['distance'] for d in data["bcenters"]]
    if bc_distances:
        min_bc_distance = np.min(bc_distances)
        mean_bc_distance = np.mean(bc_distances)
    else:
        min_bc_distance = None
        mean_bc_distance = None

    # Станции метро: число входов и кол-во станций
    metros = [d for d in data["metro_exits"].values()]
    n_metro_exits = len(metros)
    n_metro_stations = len(set([d["name"] for d in metros]))

    pd_dict = {
        'district_name': data['district_name'],
        
        f'n_buildings_{radius}m': n_buildings,
        f'n_living_buildings_{radius}m': n_living_buildings,
        f'n_flats_{radius}m': n_flats,
        f'avg_building_age_{radius}m': avg_building_age,
        f'secondary_flat_price_{radius}m': secondary_flat_price,

        f'min_bc_distance_{radius}m': min_bc_distance,
        f'mean_bc_distance_{radius}m': mean_bc_distance,

        f'n_metro_exits_{radius}m': n_metro_exits,
        f'n_metro_stations_{radius}m': n_metro_stations,

        f'traffic1_{radius}m': data['traffic1'],
        f'traffic2_{radius}m': data['traffic2'],
        f'traffic3_{radius}m': data['traffic3'],
        f'traffic4_{radius}m': data['traffic4'],
    }

    pd_dict.update(data["price"]) # Не зависит от радиуса
    pd_dict.update(orgs_dict)
    return pd_dict

In [86]:
import pandas as pd

pd.DataFrame(get_data(55.741036, 37.654256, 500), index=[0])

,district_name,n_buildings_500m,n_living_buildings_500m,n_flats_500m,avg_building_age_500m,secondary_flat_price_500m,min_bc_distance_500m,mean_bc_distance_500m,n_metro_exits_500m,n_metro_stations_500m,...,Ветаптеки и ветклиники_count_500m,Магазины цветов_count_500m,"Прачечные, химчистки_count_500m",Детские игровые залы_count_500m,Религия_count_500m,"Пиццерии, суши, столовые_count_500m","Пекарни, кофейни_count_500m","Кафе, бары, рестораны_count_500m",Социальные_count_500m,Банки_count_500m
0,Таганский,344,57,3677,84,546200.0,39,336.764151,7,2,...,2,15,13,1,4,15,30,103,2,7


In [87]:
get_data(55.741036, 37.654256, 500)

{'district_name': 'Таганский',
 'n_buildings_500m': 344,
 'n_living_buildings_500m': 57,
 'n_flats_500m': 3677,
 'avg_building_age_500m': 84,
 'secondary_flat_price_500m': 546200.0,
 'min_bc_distance_500m': np.int64(39),
 'mean_bc_distance_500m': np.float64(336.7641509433962),
 'n_metro_exits_500m': 7,
 'n_metro_stations_500m': 2,
 'traffic1_500m': 470,
 'traffic2_500m': 12666,
 'traffic3_500m': 2532,
 'traffic4_500m': 20393,
 'district_price': '433700',
 'district_price_room1': '400000',
 'district_price_room2': '428400',
 'district_price_room3': '443800',
 'district_price_room4': '503000',
 'Медицина_count_500m': 50,
 'Стоматологии_count_500m': 19,
 'Аптеки, оптики_count_500m': 35,
 'Салоны красоты_count_500m': 188,
 'Бани, сауны_count_500m': 0,
 'Фитнес-центры, тренажерные_count_500m': 5,
 'Супермаркеты_count_500m': 7,
 'Гипермаркеты_count_500m': 0,
 'Продуктовые магазины_count_500m': 26,
 'Алкомаркеты, магазины пива_count_500m': 12,
 'Школы, лицеи, гимназии_count_500m': 13,
 'Детск

## ==============================================================

In [99]:
from glob import glob
import pandas as pd

df = pd.concat([
    pd.read_csv(file)
    for file in glob("../data/geocoding_ya/*.csv")
], axis=0)[["lat", "lon"]].dropna().tail(10)
df

,lat,lon
7185,55.654952,37.618476
7188,55.658911,37.620214
7191,55.654520,37.618732
7192,55.654595,37.617918
7193,55.654595,37.617918
7194,55.657776,37.620317
7197,55.657101,37.621272
7198,55.651736,37.618845
7199,55.655788,37.620557
7200,55.655120,37.620263


In [100]:
def get_batch_data(df_batch: pd.DataFrame, radius: int):
    results_batch = []
    for i in range(df_batch.shape[0]):
        row = df_batch.iloc[i]
        results_batch.append(
            get_data(row['lat'], row['lon'], radius)
        )
    
    results_df = pd.DataFrame(results_batch).set_index(df_batch.index)
    return pd.concat([df_batch, results_df], axis=1)

In [101]:
_ = get_batch_data(df.tail(3), 300)
_

,lat,lon,district_name,n_buildings_300m,n_living_buildings_300m,n_flats_300m,avg_building_age_300m,secondary_flat_price_300m,min_bc_distance_300m,mean_bc_distance_300m,...,Религия_count_300m,"Пиццерии, суши, столовые_count_300m","Пекарни, кофейни_count_300m","Кафе, бары, рестораны_count_300m",Социальные_count_300m,Банки_count_300m,house_price,house_price_room2,house_price_room3,house_price_room4
7198,55.651736,37.618845,Нагорный,51,15,1298,59,339600.0,7,175.0,...,0,2,8,15,1,2,NaN,NaN,NaN,NaN
7199,55.655788,37.620557,Нагорный,54,26,1754,61,323100.0,152,223.2,...,0,1,6,12,1,3,NaN,NaN,NaN,NaN
7200,55.655120,37.620263,Нагорный,50,27,1888,61,327500.0,116,170.0,...,0,1,6,12,1,3,295367,312500,296200,277400


In [ ]:
# Глобальный стейт
state_df = pd.DataFrame(index=df.inde)

def get_all_data(batch_size: int, sz: int):
    for i in range(0, sz, batch_size):
        dt1 = get_batch_data(i, i + batch_size, 300)
        dt2 = get_batch_data(i, i + batch_size, 600)
        if dt1 is None or dt2 is None:
            continue
        _ = pd.concat([dt1, dt2.drop(
            ['district_name', 'district_price', 
             'district_price_room1', 'district_price_room2', 
             'district_price_room3', 'district_price_room4'
             ], axis=1)], axis=1)
        new_df = pd.concat([new_df, _], axis=0)
        new_df.to_csv("../data/moscow_super_transformed.csv")